In [2]:
# import libraries
import pandas as pd 
import numpy as np

In [3]:
# setup project root and directories
from pathlib import Path
Project_root=Path.cwd().parent
Raw_data_dir=Project_root/"data"/"raw"

print("Project root:",Project_root)
print("Raw data directory:",Raw_data_dir)
print("Exists:",Raw_data_dir.exists())

Project root: d:\certificates\zidio\project_foresight
Raw data directory: d:\certificates\zidio\project_foresight\data\raw
Exists: True


In [4]:
# check all the raw_files
raw_files=list(Raw_data_dir.glob("*.csv"))
for file in raw_files:
    print(file.name)

customer_master.csv
inventory_snapshot.csv
promotions.csv
sales_transactions.csv
sku_inventory_flags.csv
sku_master.csv
store_master.csv


In [5]:
# check size of eacch files
for file in raw_files:
    size_mb=file.stat().st_size / (1024 **2)
    print(f"{file.name:<30}{size_mb:.2f} MB")

customer_master.csv           0.53 MB
inventory_snapshot.csv        0.89 MB
promotions.csv                0.01 MB
sales_transactions.csv        763.98 MB
sku_inventory_flags.csv       0.12 MB
sku_master.csv                0.44 MB
store_master.csv              0.00 MB


In [6]:
# read all the small csv files
# expect the sales_transactions.csv file due to large size
customer=pd.read_csv(Raw_data_dir / "customer_master.csv")
inventory=pd.read_csv(Raw_data_dir / "inventory_snapshot.csv")
promotions=pd.read_csv(Raw_data_dir / "promotions.csv")
sku_flags=pd.read_csv(Raw_data_dir / "sku_inventory_flags.csv")
sku=pd.read_csv(Raw_data_dir / "sku_master.csv")
store=pd.read_csv(Raw_data_dir / "store_master.csv")

print("Customer: ",customer.shape)
print("Inventory: ", inventory.shape)
print("Promotions: ",promotions.shape)
print("SKU Flags: ",sku_flags.shape)
print("SKU: ",sku.shape)
print("Store: ",store.shape)

Customer:  (10000, 7)
Inventory:  (26408, 6)
Promotions:  (100, 8)
SKU Flags:  (600, 6)
SKU:  (5000, 7)
Store:  (30, 5)


# Customer csv

In [7]:
customer.head()

,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date
0,CUST00001,52,Female,Sukkur,Silver,Online,2021-01-01
1,CUST00002,41,Female,Karachi,Silver,Mobile App,2021-05-14
2,CUST00003,48,Female,Bahawalpur,Bronze,In-Store,2022-10-23
3,CUST00004,45,Male,Karachi,Silver,In-Store,2015-07-15
4,CUST00005,39,Female,Faisalabad,Bronze,In-Store,2024-01-02


In [8]:
customer.info()
customer.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   cust_id            10000 non-null  object
 1   age                10000 non-null  int64 
 2   gender             10000 non-null  object
 3   city               10000 non-null  object
 4   loyalty_segment    10000 non-null  object
 5   preferred_channel  10000 non-null  object
 6   registration_date  10000 non-null  object
dtypes: int64(1), object(6)
memory usage: 547.0+ KB


cust_id              object
age                   int64
gender               object
city                 object
loyalty_segment      object
preferred_channel    object
registration_date    object
dtype: object

In [9]:
customer.describe()

,age
count,10000.000000
mean,37.702900
std,12.190106
min,18.000000
25%,29.000000
50%,37.000000
75%,46.000000
max,80.000000


In [10]:
# count the unique trait of each column
print("Gender; ")
print(customer["gender"].value_counts(dropna=False))
print("\n City: ")
print(customer["city"].value_counts(dropna=False))
print("\n Loyalty Segment: ")
print(customer["loyalty_segment"].value_counts(dropna=False))
print("\n Preferred Channel: ")
print(customer["preferred_channel"].value_counts(dropna=False))

Gender; 
gender
Female    4930
Male      4885
Other      185
Name: count, dtype: int64

 City: 
city
Karachi       1577
Lahore        1418
Islamabad      996
Rawalpindi     832
Faisalabad     796
Peshawar       640
Multan         602
Sialkot        470
Gujranwala     451
Quetta         449
Sukkur         390
Hyderabad      374
Abbottabad     354
Bahawalpur     342
Sargodha       309
Name: count, dtype: int64

 Loyalty Segment: 
loyalty_segment
Bronze      3969
Silver      3106
Gold        1905
Platinum    1020
Name: count, dtype: int64

 Preferred Channel: 
preferred_channel
In-Store      5512
Online        2994
Mobile App    1494
Name: count, dtype: int64


In [11]:
# check whether the cust_id is primary key or not
# must have missing_values and duplicates as zero
print("Total rows: ", len(customer))
print("Unique customer IDs: ",customer["cust_id"].nunique())
print("Duplicate customer IDs: ",customer["cust_id"].duplicated().sum())
print("Missing customer IDs: ",customer["cust_id"].isna().sum())

Total rows:  10000
Unique customer IDs:  10000
Duplicate customer IDs:  0
Missing customer IDs:  0


In [12]:
customer.isnull().sum()

cust_id              0
age                  0
gender               0
city                 0
loyalty_segment      0
preferred_channel    0
registration_date    0
dtype: int64

In [13]:
customer.duplicated().sum()

np.int64(0)

In [14]:
# registration_date is stored as object (string); 
# convert to datetime during the data cleaning phase if required.
customer["registration_date"].describe()

count          10000
unique          3661
top       2025-02-02
freq              10
Name: registration_date, dtype: object

# Summary till now
### Customer Master Dataset

### Grain
- One row represents one customer.

### Candidate Primary Key
- cust_id

### Initial Observations
- 10,000 records and 7 columns.
- No missing values.
- No duplicate rows.
- registration_date is stored as object (string).
- Age ranges from 18 to 80.
- Customer attributes include age, gender, city, loyalty segment, and preferred shopping channel.

# now Inventory csv


In [15]:
inventory.head()

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date
0,ST11,SKU02558,333,72,22,2025-06-23
1,ST21,SKU01031,236,67,14,2025-06-17
2,ST26,SKU02129,496,96,34,2025-07-10
3,ST19,SKU02907,109,34,13,2025-08-17
4,ST05,SKU01023,333,97,22,2025-07-12


In [16]:
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26408 entries, 0 to 26407
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   store_id           26408 non-null  object
 1   sku_id             26408 non-null  object
 2   stock_on_hand      26408 non-null  int64 
 3   reorder_point      26408 non-null  int64 
 4   safety_stock       26408 non-null  int64 
 5   last_restock_date  26408 non-null  object
dtypes: int64(3), object(3)
memory usage: 1.2+ MB


In [17]:
inventory.describe()

,stock_on_hand,reorder_point,safety_stock
count,26408.000000,26408.000000,26408.000000
mean,166.195812,59.983301,20.951719
std,142.238780,23.389128,9.873781
min,0.000000,20.000000,4.000000
25%,40.000000,40.000000,13.000000
50%,140.000000,60.000000,20.000000
75%,264.000000,80.000000,28.000000
max,598.000000,100.000000,50.000000


In [18]:
inventory.nunique()

store_id               30
sku_id               4495
stock_on_hand         595
reorder_point          81
safety_stock           47
last_restock_date     221
dtype: int64

In [19]:
print("Missing Values")
print(inventory.isnull().sum())

print("\nDuplicate Rows")
print(inventory.duplicated().sum())

print("\nUnique Stores")
print(inventory["store_id"].nunique())

print("\nUnique SKUs")
print(inventory["sku_id"].nunique())

Missing Values
store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
dtype: int64

Duplicate Rows
0

Unique Stores
30

Unique SKUs
4495


In [20]:
inventory.duplicated(subset=["store_id","sku_id"]).sum()

np.int64(0)

# Inventory Snapshot Dataset

### Grain
- One row represents the inventory status of one SKU at one store.

### Candidate Primary Key
- Composite key: (store_id, sku_id)
- No duplicate store-SKU combinations were found.

### Possible Foreign Keys
- store_id → store_master
- sku_id → sku_master

### Initial Observations
- 26,408 records and 6 columns.
- Contains inventory measures: stock_on_hand, reorder_point, and safety_stock.
- No standard missing values were found.
- No fully duplicate rows were found.
- Contains 30 unique stores and 4,495 unique SKUs.
- stock_on_hand ranges from 0 to 598; zero stock may represent an out-of-stock condition and should be investigated later.
- last_restock_date is currently stored as object rather than datetime.

# Promotions


In [21]:
promotions.head()

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value
0,PROMO001,Spring Discount Days,2024-07-28,2024-08-02,15.1,BOGO,Brand,NovaFresh
1,PROMO002,Summer Savings Event,2023-03-09,2023-03-21,7.0,Bundle Offer,All,All
2,PROMO003,Payday Discount Days,2025-04-30,2025-05-26,10.2,BOGO,Brand,ZestyCo
3,PROMO004,Super Sale,2025-06-07,2025-06-17,15.9,Clearance,SKU,SKU01108
4,PROMO005,Festive Discount Days,2024-03-23,2024-04-08,46.0,Bundle Offer,All,All


In [22]:
promotions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   promo_id      100 non-null    object 
 1   promo_name    100 non-null    object 
 2   start_date    100 non-null    object 
 3   end_date      100 non-null    object 
 4   discount_pct  100 non-null    float64
 5   promo_type    100 non-null    object 
 6   target_type   100 non-null    object 
 7   target_value  100 non-null    object 
dtypes: float64(1), object(7)
memory usage: 6.4+ KB


In [23]:
promotions.describe()

,discount_pct
count,100.000000
mean,27.580000
std,14.134348
min,5.000000
25%,15.400000
50%,25.400000
75%,42.575000
max,49.800000


In [24]:
print("Promotion Types: ")
print(promotions["promo_type"].value_counts(dropna=False))

print("\nTarget Types: ")
print(promotions["target_type"].value_counts(dropna=False))

Promotion Types: 
promo_type
Bundle Offer           23
Percentage Discount    22
Clearance              20
Flat Discount          18
BOGO                   17
Name: count, dtype: int64

Target Types: 
target_type
Category    38
SKU         22
Brand       21
All         19
Name: count, dtype: int64


In [25]:
for target in promotions["target_type"].unique():
    print(f"\nTarget Type: {target}")
    print(promotions.loc[promotions["target_type"]==target,"target_value"].value_counts())


Target Type: Brand
target_value
ValueChoice     3
CleanWave       3
NovaFresh       2
MetroBrand      2
ZestyCo         1
BasicNeeds      1
HomeStyle       1
EverydayPlus    1
TrueTaste       1
PrimeCare       1
SmartBuy        1
SparkleClean    1
DailyBest       1
EliteHome       1
UrbanBite       1
Name: count, dtype: int64

Target Type: All
target_value
All    19
Name: count, dtype: int64

Target Type: SKU
target_value
SKU01108    1
SKU04041    1
SKU01318    1
SKU04487    1
SKU01969    1
SKU04841    1
SKU02950    1
SKU02302    1
SKU01797    1
SKU02406    1
SKU02463    1
SKU01028    1
SKU00800    1
SKU01012    1
SKU02021    1
SKU02605    1
SKU01419    1
SKU00089    1
SKU01676    1
SKU02155    1
SKU03430    1
SKU04353    1
Name: count, dtype: int64

Target Type: Category
target_value
Beverages                    5
Stationery & Office          5
Grocery                      4
Apparel & Footwear           4
Home & Kitchen               4
Health & Wellness            3
Frozen Foods     

In [26]:
print("Missing values: ")
print(promotions.isnull().sum())

print("\nDuplicate Values: ")
print(promotions.duplicated().sum())

print("\nUnique Promotions IDs: ")
print(promotions["promo_id"].nunique())

print("\nDuplicate promotions IDs: ")
print(promotions["promo_id"].duplicated().sum())

Missing values: 
promo_id        0
promo_name      0
start_date      0
end_date        0
discount_pct    0
promo_type      0
target_type     0
target_value    0
dtype: int64

Duplicate Values: 
0

Unique Promotions IDs: 
100

Duplicate promotions IDs: 
0


# Promotions Dataset

### Grain
- One row represents one promotion/campaign.

### Candidate Primary Key
- promo_id
- All 100 promotion IDs are unique.

### Promotion Targeting
- Promotions can target different levels: Category, SKU, Brand, or All.
- target_value identifies the specific category, SKU, or brand depending on target_type.
- When target_type is All, target_value represents All.

### Initial Observations
- 100 records and 8 columns.
- No standard missing values were found.
- No fully duplicate rows were found.
- discount_pct ranges from 5.0 to 49.8.
- start_date and end_date are currently stored as object rather than datetime.
- Five promotion types are present: Bundle Offer, Percentage Discount, Clearance, Flat Discount, and BOGO.

# SKU inventory Flag

In [27]:
sku_flags.head()

,sku_id,flag,affected_stores,window_start,window_end,notes
0,SKU04321,STOCKOUT_RISK,ST22;ST26;ST23;ST15;ST12;ST28;ST01;ST27;ST08;S...,2025-11-06,2025-12-03,Top-selling SKU (by observed volume); injected...
1,SKU04596,STOCKOUT_RISK,ST12;ST09;ST21;ST23;ST07;ST24;ST01;ST28;ST14;S...,2025-10-25,2025-11-06,Top-selling SKU (by observed volume); injected...
2,SKU03727,STOCKOUT_RISK,ST19;ST14;ST28;ST29;ST02;ST07;ST17;ST08;ST05;ST16,2025-11-08,2025-11-29,Top-selling SKU (by observed volume); injected...
3,SKU04154,STOCKOUT_RISK,ST15;ST03;ST30;ST09;ST24;ST19;ST28;ST02;ST07;S...,2025-11-12,2025-11-26,Top-selling SKU (by observed volume); injected...
4,SKU00953,STOCKOUT_RISK,ST09;ST19;ST27;ST20;ST01;ST03;ST11;ST10;ST22;S...,2025-11-29,2025-12-25,Top-selling SKU (by observed volume); injected...


In [28]:
sku_flags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sku_id           600 non-null    object
 1   flag             600 non-null    object
 2   affected_stores  600 non-null    object
 3   window_start     200 non-null    object
 4   window_end       200 non-null    object
 5   notes            600 non-null    object
dtypes: object(6)
memory usage: 28.3+ KB


In [29]:
# find unique number of elements in each row
sku_flags.nunique()

sku_id             600
flag                 2
affected_stores    600
window_start        60
window_end          62
notes                2
dtype: int64

In [30]:
# check duplicates in sku_id column
sku_flags["sku_id"].duplicated().sum()

np.int64(0)

In [31]:
sku_flags["flag"].value_counts()

flag
SLOW_MOVER       400
STOCKOUT_RISK    200
Name: count, dtype: int64

In [32]:
sku_flags.groupby("flag")[["window_start","window_end"]].count()

,window_start,window_end
flag,,
SLOW_MOVER,0,0
STOCKOUT_RISK,200,200


In [33]:
sku_flags.groupby("flag")["notes"].unique()

flag
SLOW_MOVER       [Bottom-selling SKU (by observed volume); inje...
STOCKOUT_RISK    [Top-selling SKU (by observed volume); injecte...
Name: notes, dtype: object

In [34]:
for note in sku_flags["notes"].unique():
    print(note)

Top-selling SKU (by observed volume); injected recent stockout window.
Bottom-selling SKU (by observed volume); injected overstock + stale restock date.


In [35]:
sku_flags["affected_stores"].str.split(";").str.len().value_counts().sort_index()

affected_stores
9     22
10    20
11    20
12    16
13    15
14    20
15    60
16    42
17    39
18    45
19    27
20    26
21    20
22    29
23    29
24    27
25    27
26    21
27    28
28    25
29    24
30    18
Name: count, dtype: int64

In [36]:
all_affected_stores=sku_flags["affected_stores"].str.split(";").explode()
all_affected_stores.nunique()

30

In [37]:
all_affected_stores.isin(store["store_id"]).value_counts()

affected_stores
True    11649
Name: count, dtype: int64

In [38]:
sku_flags["sku_id"].isin(sku["sku_id"]).value_counts()

sku_id
True    600
Name: count, dtype: int64

In [39]:
sku_flags.duplicated().sum()

np.int64(0)

In [40]:
pd.to_datetime(sku_flags["window_start"]).agg(["min", "max"])

min   2025-10-17
max   2025-12-17
Name: window_start, dtype: datetime64[ns]

In [41]:
pd.to_datetime(sku_flags["window_end"]).agg(["min", "max"])


min   2025-10-29
max   2025-12-31
Name: window_end, dtype: datetime64[ns]

In [42]:
(pd.to_datetime(sku_flags["window_end"]) < 
 pd.to_datetime(sku_flags["window_start"])).sum()

np.int64(0)

In [43]:
sku_flags.isnull().sum()

sku_id               0
flag                 0
affected_stores      0
window_start       400
window_end         400
notes                0
dtype: int64

In [44]:
(sku_flags.select_dtypes("object")
          .apply(lambda col: col.str.strip().eq("").sum()))

sku_id             0
flag               0
affected_stores    0
window_start       0
window_end         0
notes              0
dtype: int64

In [45]:
(pd.to_datetime(sku_flags["window_end"]) -
 pd.to_datetime(sku_flags["window_start"])).dt.days.describe()

count    200.00000
mean      19.56500
std        5.99378
min       10.00000
25%       14.00000
50%       20.00000
75%       25.00000
max       29.00000
dtype: float64

In [49]:
sku_flags.assign(
    store_count=sku_flags["affected_stores"].str.split(";").str.len()
).groupby("flag")["store_count"].describe()

,count,mean,std,min,25%,50%,75%,max
flag,,,,,,,,
SLOW_MOVER,400.0,22.365,4.521935,15.0,19.0,22.0,26.0,30.0
STOCKOUT_RISK,200.0,13.515,2.888162,9.0,11.0,14.0,16.0,18.0


In [50]:
sku_flags.groupby("sku_id")["flag"].count().value_counts()

flag
1    600
Name: count, dtype: int64

## Dataset: sku_inventory_flags

### Grain
- One row represents one flagged SKU.

### Dimensions
- Rows: 600
- Columns: 6

### Candidate Primary Key
- `sku_id`
- Verified unique (600 unique values, no duplicates).

### Candidate Foreign Keys
- `sku_id` → `sku_master.sku_id` (verified)
- `affected_stores` contains semicolon-separated store IDs that all exist in `store_master.store_id` (verified)

### Column Summary
| Column | Description |
|---------|-------------|
| sku_id | SKU identifier |
| flag | Inventory flag type |
| affected_stores | Semicolon-separated list of affected store IDs |
| window_start | Start of stockout window (only for STOCKOUT_RISK) |
| window_end | End of stockout window (only for STOCKOUT_RISK) |
| notes | Explanation of why the SKU was flagged |

### Data Types
- All columns are currently stored as `object`.
- `window_start` and `window_end` are date-like strings.

### Missing Values
- `window_start`: 400 missing
- `window_end`: 400 missing
- Other columns: no standard missing values

The missing date values are structural:
- `STOCKOUT_RISK` rows contain both dates.
- `SLOW_MOVER` rows have both dates missing.

### Duplicate Check
- No duplicate rows.
- `sku_id` is unique.

### Flag Distribution
- `SLOW_MOVER`: 400 SKUs
- `STOCKOUT_RISK`: 200 SKUs

### Notes
- `STOCKOUT_RISK`: Top-selling SKU with an injected recent stockout window.
- `SLOW_MOVER`: Bottom-selling SKU with injected overstock and stale restock date.

### Date Window Observations
- `window_start`: 2025-10-17 to 2025-12-17
- `window_end`: 2025-10-29 to 2025-12-31
- No cases where `window_end` occurs before `window_start`.

### Affected Stores
- Each flagged SKU affects between **9 and 30 stores**.
- Across all rows, 30 unique stores are referenced.
- All referenced stores exist in `store_master`.

### Observations
- Each flagged SKU has exactly one inventory flag.
- `SLOW_MOVER` SKUs generally affect more stores than `STOCKOUT_RISK` SKUs (mean ≈22 vs ≈14 affected stores).
- The dataset appears to contain intentionally injected inventory scenarios for analysis.

# Sku file

In [52]:
sku.shape

(5000, 7)

In [53]:
sku.head()

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,SKU00004,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,SKU00005,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [59]:
sku.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku_id       5000 non-null   object 
 1   sku_name     5000 non-null   object 
 2   category     5000 non-null   object 
 3   subcategory  5000 non-null   object 
 4   unit_price   5000 non-null   float64
 5   cost_price   5000 non-null   float64
 6   brand        5000 non-null   object 
dtypes: float64(2), object(5)
memory usage: 273.6+ KB


In [54]:
sku.describe()

,unit_price,cost_price
count,5000.000000,5000.000000
mean,619.191672,417.040736
std,708.009319,480.347009
min,24.410000,16.750000
25%,160.555000,108.825000
50%,318.745000,214.670000
75%,788.715000,530.027500
max,4755.470000,3398.160000


In [55]:
sku["sku_id"].nunique()

5000

In [57]:
sku["sku_id"].is_unique

True

In [58]:
sku["sku_id"].duplicated().sum()

np.int64(0)

In [60]:
sku[["category","subcategory","brand"]].nunique()

category       12
subcategory    53
brand          30
dtype: int64

In [62]:
print(sorted(sku["category"].unique()))

['Apparel & Footwear', 'Beverages', 'Dairy & Bakery', 'Electronics & Accessories', 'Frozen Foods', 'Grocery', 'Health & Wellness', 'Home & Kitchen', 'Home Care', 'Personal Care', 'Snacks & Confectionery', 'Stationery & Office']


In [63]:
print(sorted(sku["subcategory"].unique()))


['Air Fresheners', 'Bath & Body', 'Batteries', 'Biscuits & Cookies', 'Bottled Water', 'Bread', 'Butter & Ghee', 'Candy', 'Cheese', 'Chips & Crisps', 'Chocolates', 'Coffee', 'Cooking Oil', 'Cookware', 'Dishwashing', 'Eggs', 'Energy Drinks', 'First Aid', 'Flour & Atta', 'Footwear', 'Frozen Meat', 'Frozen Vegetables', 'Hair Care', 'Home Decor', 'Ice Cream', 'Juices', 'Kids Wear', 'Laundry', "Men's Grooming", "Men's Wear", 'Milk', 'Mobile Accessories', 'Namkeen', 'Notebooks', 'OTC Medicine', 'Office Supplies', 'Oral Care', 'Pest Control', 'Pulses & Lentils', 'Ready Meals', 'Rice & Grains', 'Skin Care', 'Small Appliances', 'Soft Drinks', 'Spices & Seasoning', 'Storage & Containers', 'Sugar & Salt', 'Surface Cleaners', 'Tea', 'Vitamins & Supplements', "Women's Wear", 'Writing Instruments', 'Yogurt']


In [64]:
print(sorted(sku["brand"].unique()))


['BasicNeeds', 'CityFresh', 'CleanWave', 'CrispKing', 'DailyBest', 'EliteHome', 'EverydayPlus', 'FamilyChoice', 'FreshFarm', 'GoldenHarvest', 'GreenValley', 'HealthFirst', 'HomeStyle', 'MetroBrand', 'NatureBlend', 'NovaFresh', 'NutriPlus', 'PremiumSelect', 'PrimeCare', 'PureLife', 'PurePlus', 'QuickBite', 'SmartBuy', 'SoftTouch', 'SparkleClean', 'SunriseFoods', 'TrueTaste', 'UrbanBite', 'ValueChoice', 'ZestyCo']


In [65]:
sku["category"].value_counts()

category
Stationery & Office          459
Beverages                    457
Dairy & Bakery               428
Home Care                    425
Frozen Foods                 422
Snacks & Confectionery       414
Apparel & Footwear           410
Personal Care                410
Health & Wellness            396
Grocery                      395
Electronics & Accessories    394
Home & Kitchen               390
Name: count, dtype: int64

In [66]:
sku["brand"].value_counts()

brand
CleanWave        193
GreenValley      190
TrueTaste        185
NutriPlus        184
FreshFarm        184
MetroBrand       183
EliteHome        180
QuickBite        179
CityFresh        177
PurePlus         176
FamilyChoice     175
EverydayPlus     173
NovaFresh        170
SoftTouch        169
DailyBest        168
UrbanBite        167
SparkleClean     166
ZestyCo          163
PremiumSelect    161
SmartBuy         160
GoldenHarvest    160
PrimeCare        155
HealthFirst      155
HomeStyle        153
BasicNeeds       150
ValueChoice      148
CrispKing        146
NatureBlend      146
PureLife         143
SunriseFoods     141
Name: count, dtype: int64

In [68]:
sku.groupby("subcategory")["category"].nunique().value_counts()

category
1    53
Name: count, dtype: int64

In [69]:
sku[["unit_price","cost_price"]].describe()

,unit_price,cost_price
count,5000.000000,5000.000000
mean,619.191672,417.040736
std,708.009319,480.347009
min,24.410000,16.750000
25%,160.555000,108.825000
50%,318.745000,214.670000
75%,788.715000,530.027500
max,4755.470000,3398.160000


In [72]:
# check if all the items have unit price more than cost_price
(sku["unit_price"]>sku["cost_price"]).sum()

np.int64(5000)

In [74]:
sku.duplicated().sum()

np.int64(0)

In [77]:
# check all the ids
(~sku_flags["sku_id"].isin(sku["sku_id"])).sum()

np.int64(0)

## SKU Master - Data Understanding Summary

### Table Overview
- Rows: **5,000**
- Columns: **7**
- Memory Usage: **273.6 KB**

### Grain
- One row represents **one unique Stock Keeping Unit (SKU).**

### Candidate Primary Key
- `sku_id`
- **Verified:** Unique (5,000 distinct values), no duplicates.

### Columns
| Column | Data Type | Missing Values |
|---------|-----------|----------------|
| sku_id | object | 0 |
| sku_name | object | 0 |
| category | object | 0 |
| subcategory | object | 0 |
| unit_price | float64 | 0 |
| cost_price | float64 | 0 |
| brand | object | 0 |

### Data Quality
- No missing values.
- No duplicate rows.
- All price values are populated.

### Product Hierarchy
- 12 product categories.
- 53 subcategories.
- **Verified:** Every subcategory belongs to exactly one category (one-to-many relationship from category to subcategory).

### Pricing
- Unit Price Range: **24.41 – 4755.47**
- Cost Price Range: **16.75 – 3398.16**
- **Verified:** Every SKU has `unit_price > cost_price`.

### Candidate Relationships
- `sku_id` → Primary Key
- **Verified:** `sku_inventory_flags.sku_id` references valid `sku_master.sku_id` values.

# store master

In [91]:
store.head()

,store_id,store_name,city,store_type,opening_date
0,ST01,Quetta Convenience Store #01,Quetta,Convenience Store,2017-07-08
1,ST02,Islamabad Express Store #02,Islamabad,Express Store,2019-05-08
2,ST03,Sialkot Supermarket #03,Sialkot,Supermarket,2019-03-28
3,ST04,Multan Supermarket #04,Multan,Supermarket,2017-10-08
4,ST05,Karachi Supermarket #05,Karachi,Supermarket,2020-11-03


In [83]:
store.shape

(30, 5)

In [85]:
store["store_id"].is_unique

True

In [88]:
store.describe()

,store_id,store_name,city,store_type,opening_date
count,30,30,30,30,30
unique,30,30,12,4,30
top,ST01,Quetta Convenience Store #01,Karachi,Convenience Store,2017-07-08
freq,1,1,5,11,1


In [89]:
store.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   store_id      30 non-null     object
 1   store_name    30 non-null     object
 2   city          30 non-null     object
 3   store_type    30 non-null     object
 4   opening_date  30 non-null     object
dtypes: object(5)
memory usage: 1.3+ KB


In [92]:
store["city"].value_counts()

city
Karachi       5
Islamabad     4
Multan        4
Quetta        3
Sialkot       3
Sukkur        2
Peshawar      2
Rawalpindi    2
Lahore        2
Hyderabad     1
Faisalabad    1
Gujranwala    1
Name: count, dtype: int64

In [93]:
store["store_type"].value_counts()

store_type
Convenience Store    11
Supermarket          11
Hypermarket           7
Express Store         1
Name: count, dtype: int64

In [96]:
store["opening_date"].min(), store["opening_date"].max()

('2014-04-01', '2021-12-10')

In [100]:
(inventory["store_id"].isin(store["store_id"])).sum()

np.int64(26408)

## Store Master - Data Understanding Summary

### Table Overview
- Rows: **30**
- Columns: **5**

### Grain
- One row represents **one retail store**.

### Candidate Primary Key
- `store_id`
- **Verified:** Unique (30 distinct values), no duplicates.

### Columns

| Column | Data Type | Missing Values |
|---------|-----------|----------------|
| store_id | object | 0 |
| store_name | object | 0 |
| city | object | 0 |
| store_type | object | 0 |
| opening_date | object | 0 |

### Data Quality
- No missing values.
- No duplicate rows.
- `opening_date` is currently stored as **object (string)** in ISO format (`YYYY-MM-DD`).

### Cities
- **12 unique cities**.
- Karachi has the highest number of stores (5).
- Three cities have one store each.

### Store Types
- Convenience Store: 11
- Supermarket: 11
- Hypermarket: 7
- Express Store: 1

### Opening Period
- Earliest opening date: **2014-04-01**
- Latest opening date: **2021-12-10**

### Candidate Relationships
- Primary Key: `store_id`
- **Verified:** `inventory_snapshot.store_id` references valid `store_master.store_id` values.
- **Verified:** All store IDs referenced in `sku_inventory_flags.affected_stores` exist in `store_master`.